# MACURA vs MBPO vs M2AC vs SAC: a racing drone

Reinforcement-learning algorithms learn the same drone race from scratch. They share everything (policy
learner SAC, world-model ensemble, data, evaluation scenarios) except **how they use the learned world model
to imagine extra practice**:

| algorithm | imagined practice |
|---|---|
| **MACURA** | imagines ahead from real states and **stops each imagined trip where its models disagree** |
| MBPO | always imagines 10 steps ahead, no check |
| M2AC | imagines 10 steps, keeps only its 25% least uncertain imagined steps |
| SAC | no world model: learns from real flights only |

The question (paper: *Trust the Model Where It Trusts Itself*, Frauenknecht et al., ICML 2024): does
MACURA learn faster and break fewer drones because it does **not** trust its model everywhere?

**The final study** (decided before running it): seeds **1 and 2**, **100,000** real steps each, **MACURA, MBPO and
SAC**; about 9 h per Kaggle session, one seed per session. Seed 0 (50,000 steps, all four) is the pilot. M2AC is
left out of the long runs because all four at 100k would not fit in the GPU budget (~25 h) or in one 12 h session;
MBPO is the method MACURA builds on, so MACURA vs MBPO is the main comparison.

**How to run.** Set `SEED = 1`, push, wait; then `SEED = 2` (another session). Section 8 merges the seeds
it finds in the session's inputs; or run `python compare.py out_seed1 out_seed2` on your computer. Watch the
trained drones live on a Mac with `mjpython simulate.py` (README.md).

1. Setup · 2. The task · 3. The protocol · 4. Training and test of each algorithm · 5. Results (this seed) ·
6. Does MACURA trust its model 100%? · 7. Watch them race · 8. All seeds · 9. Outputs

## 1. Setup

In [ ]:
import os, sys, time, subprocess

SEED   = 1                       # one seed per Kaggle session: 1, then 2 (seed 0 = the 50k pilot)
STEPS  = 100000                  # real environment steps per algorithm (2000 = a quick test of the notebook)
ALGOS  = "macura mbpo sac"       # ~4.5 h + 3.4 h + 0.5 h at 100k (estimated from the seed-0 timings)
VIDEOS = True                    # test videos after each algorithm + the race videos (~25 min in total)
REPO, BRANCH = "silvergjeka22/macura-drone", "claude/bold-hawking-f0m819"
ROOT = "/kaggle/working/macura-drone"

T0 = time.time()                 # Kaggle stops a session after 12 h and then saves nothing: training stops at
DEADLINE = T0 + 11.0 * 3600      # 11 h (a run still training is cut there and saved), videos only if time is left
os.environ.update({"MACURA_TASK": "race2", "MACURA_STEPS": str(STEPS), "MACURA_SEEDS": str(SEED),
                   "MACURA_ALGOS": ALGOS, "MACURA_DEADLINE": str(DEADLINE)})

token = os.environ.get("GITHUB_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)
sys.path.insert(0, ROOT)

In [ ]:
from src.bootstrap import setup
setup(root=ROOT)

import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, Video, display
import src.config.config as cfg
from src.training.seed import set_seed
from src.training.train import train_algo
from src.analysis import results as R, report
from src.viz import figures as F, video as V

set_seed(cfg.SEED)
OUT = cfg.OUTPUT_ROOT
REFS = R.autopilot_reference(cfg.ENV, seed_base=100, episodes=20, cache=f"{OUT}/autopilot_reference.json")
RUNS = {}


def hours_left():
    return (T0 + 11.75 * 3600 - time.time()) / 3600


def trust_of(run):
    # MACURA's own saved world model + its threshold kappa at that checkpoint (for the trust panel)
    ens = R.ensemble_checkpoint(run)
    return {"ensemble": ens, "kappa": R.kappa_at(run)} if run["algo"] == "macura" and ens else None


def train(algo):
    # train one algorithm for this seed (a finished run is reloaded), then test its best checkpoint
    if algo not in cfg.ALGORITHMS:
        print(f"{algo} is not in ALGOS: skipped")
        return
    if time.time() > DEADLINE and not os.path.exists(f"{OUT}/logs/{algo}_seed{SEED}.json"):
        print(f"{algo}: no session time left, skipped")
        return
    RUNS[algo] = train_algo(algo, cfg.CFG, OUT)
    F.plot_algo(RUNS[algo], REFS, f"{OUT}/plots/{algo}_curve.png"); plt.show()
    for r in RUNS[algo]:
        s = R.run_scores(r)
        stop = r["setup"].get("stopped_early_at")
        print(f"{R.NAMES[algo]} seed {r['seed']}: avg return while learning {s['avg_return']:.0f} | "
              f"drones broken after warm-up {s['broken']:.0f} | best checkpoint at step {s['best_step']}"
              + (f" | stopped by the time limit at step {stop}" if stop else ""))
        print(f"  TEST (best checkpoint, 30 fresh scenarios): return {s['test_return']:.0f}, "
              f"success {s['test_success']:.0%} of a lap, crash rate {s['test_crash']:.0%}")
    if VIDEOS and hours_left() > 0.5:
        best = R.best_run(RUNS[algo])
        label = R.NAMES[algo]
        t = trust_of(best)
        path = V.record_race(cfg.CFG, {label: R.checkpoint(best)}, f"{OUT}/videos/test_{algo}.mp4",
                             seeds=(1000, 1001, 1002), trust={label: t} if t else None, cols=1,
                             frame_step=3, fps=17)
        if path:
            display(Video(path, embed=True, width=480))

## 2. The task
A 6 x 6 m course flown clockwise through four gates: **P1 2 m → P2 4 m → the chute (an almost vertical 3 m
drop) → P3 1 m → P4 1.5 m** (24.3 m per lap), with two pillars. Each flight lasts 10 s from a random point
of the course, with a package of random weight (0 to 0.2 kg) and a steady wind (up to 0.5 N), both
visible to the drone, plus hidden gusts and rotor noise.

- **Flying**: an on-board stabiliser in altitude-hold mode, like a consumer drone. Actions = sideways
  acceleration x, y (tilt up to 40°), climb rate (±3 m/s) and yaw rate. A racing-line sensor gives the
  vector to the nearest point of the line and the direction 0.5 m ahead.
- **Reward**: speed along the course while near it, minus the distance outside a 0.35 m tube; no finish
  line, so faster = more reward. A crash (flip, ground, pillar, more than 2.5 m off the course) ends the
  flight and costs 500.
- **The trap**: sinking fast straight down while the rotors push loses lift (vortex ring state, up to 35%
  at 2.2 m/s). Dropping the chute too fast crashes. The learned models are wrong there **and disagree about
  it** (measured offline: 6x the disagreement, 5x the error of normal flight), which is exactly what MACURA
  can detect. In the 50k pilot no policy flew fast enough to meet it (fastest descent 1.4 m/s): the longer
  runs are meant to get there.

**Success** = how much of a full lap (24.3 m) a drone flies in its 10 s flight, 0-100%; a crash ends the flight,
so it counts how far it got. Reference, a hand-written autopilot (not learned, same stabiliser): 53% at 1.5 m/s,
93% at 3 m/s, no crashes. The learning curves show these two as lines.

In [ ]:
F.plot_race_course(cfg.CFG, f"{OUT}/plots/task_course.png")
F.plot_lift_loss(cfg.CFG, f"{OUT}/plots/task_lift_loss.png")
F.plot_wind(cfg.CFG, save_path=f"{OUT}/plots/task_wind.png")
print(REFS)

In [ ]:
if VIDEOS:   # the autopilot three ways on the same flight: careful, fast, chute dive (loses lift)
    demo = V.record_race(cfg.CFG, {"careful 1.5 m/s": "autopilot:0.8:1.5", "fast 3 m/s": "autopilot:1.1:3.0",
                                   "chute dive": "autopilot:3.0:3.0"},
                         f"{OUT}/videos/task_demo.mp4", seeds=(6027,), cols=3,
                         notes={"careful 1.5 m/s": "hand-written autopilot (not learned)",
                                "fast 3 m/s": "hand-written autopilot (not learned)",
                                "chute dive": "drops the chute at 3 m/s: loses lift"})
    if demo:
        display(Video(demo, embed=True, width=960))

## 3. The protocol (fixed before the runs)
The MACURA paper's setup (App. D), the same for every algorithm:

- **World model** (MACURA, MBPO, M2AC): 7 probabilistic networks, the 5 best on held-out data used; retrained
  every 250 real steps; 25,000 imagined trips started from real states each time; each SAC batch = 5% real +
  95% imagined data; imagined data expires after 4 model rounds (as in the MACURA authors' code).
- **MACURA**: trip stops once the ensemble disagreement (GJS) exceeds κ = ξ x running mean of the 95% quantile
  of first-step disagreement (ξ = 0.5); SAC updates per real step follow how much it kept (paper Eq. 22, up to 16).
  MBPO: 10-step trips (ramped in over the first 2,500 learning steps), 8 updates. M2AC: paper mode, 8 updates.
  SAC: 1 update per real step.
- **Exploration** (as in the paper, a known confound): MACURA pink noise, MBPO and M2AC deterministic, SAC samples
  its policy. Everyone starts with 5,000 random steps.
- **Evaluation**: every 1,000 steps the greedy policy flies 20 fixed scenarios (seeds 100-119); the best one is
  saved and finally flies 30 **fresh** test scenarios (seeds 1000-1029), never used to pick it.
- **Scores**, main ones first: average return while learning, drones broken after the warm-up; then final test
  return, success (% of a lap per flight) and crash rate. With 2 seeds: their mean and both values (a confidence
  interval needs 3 or more).

Why each setting has its value, and every earlier run: `docs/EXPERIMENTS.md`.

## 4. Training and test of each algorithm
One cell per algorithm (at 100k: MACURA ~4.5 h, MBPO ~3.4 h, SAC ~0.5 h). Every 1,000 steps a line shows the
evaluation return, crash rate and drones broken so far, plus what the world model did (MACURA: trip length and
how much it trusts fast descents vs normal flight; MBPO: how much of its imagined data MACURA's rule would reject).
When an algorithm is done, its best checkpoint is **tested** on 30 fresh scenarios and flies 3 of them in a test
video (MACURA's panel shows its world model's verdict at every step). Running a cell again reloads the run.

### 4.1 MACURA

In [ ]:
train("macura")

### 4.2 MBPO

In [ ]:
train("mbpo")

### 4.3 M2AC (only in the 50k pilot; skipped unless listed in ALGOS)

In [ ]:
train("m2ac")

### 4.4 SAC

In [ ]:
train("sac")

## 5. Results for this seed
Paper Fig. 4 for this task: return while learning (dotted lines = the autopilot), crash rate, success (% of a lap
per flight) and real crashes while learning (grey = random warm-up). A single evaluation (20 flights) is noisy, so
the curves show a rolling mean over 5 evaluations with the raw values faint behind; the scores use the raw values.
Then the results table and the scores. One seed shows a trend; section 8 combines the seeds.

In [ ]:
runs = [r for a in R.ALGOS for r in RUNS.get(a, [])]
F.plot_learning_curves(runs, REFS, f"{OUT}/plots/learning_curves.png"); plt.show()
F.plot_summary_table(runs, f"{OUT}/plots/summary_table.png"); plt.show()
F.plot_scores(runs, f"{OUT}/plots/scores.png"); plt.show()
display(Markdown(R.summary_markdown(runs)))

## 6. Does MACURA trust its world model 100%?
No, and these figures show where it stops trusting it.

- **During training**: the share of each full 10-step imagination MACURA keeps (MBPO keeps 100%), MACURA's trust
  in fast descents vs normal flight, trip length and SAC updates per step.
- **κ over training** (paper Fig. 8): the threshold and this round's disagreement.
- **On real flights** (paper Fig. 10): the best policy flies 10 fresh scenarios; at every step its saved world
  model's disagreement is compared with the model's actual one-step error. Steps above κ are the ones MACURA
  would not imagine; if the disagreement is a good signal, the model is more wrong there. MBPO's best world model
  is judged with MACURA's κ for comparison.
- **The data each method learns from**: how much of MBPO's imagined training data MACURA's rule would reject, and
  how much of it is fast descents.

In [ ]:
F.plot_model_trust(runs, f"{OUT}/plots/model_trust.png"); plt.show()
print(R.trust_summary(runs))
if RUNS.get("macura"):
    F.plot_kappa(runs, f"{OUT}/plots/kappa.png"); plt.show()

In [ ]:
CHECKS = []
for algo in ("macura", "mbpo"):
    if RUNS.get(algo) and R.ensemble_checkpoint(R.best_run(RUNS[algo])):
        c = R.model_check(R.best_run(RUNS[algo]), cfg.CFG)
        if c["kappa"] is None and RUNS.get("macura"):          # MBPO: judged with MACURA's threshold
            c["kappa"] = R.kappa_at(R.best_run(RUNS["macura"]))
            c["trusted"] = float((c["gjs"] < c["kappa"]).mean())
        CHECKS.append(c)
if CHECKS:
    F.plot_model_check(CHECKS, f"{OUT}/plots/model_check.png"); plt.show()
F.plot_untrusted_data(runs, f"{OUT}/plots/untrusted_data.png"); plt.show()
F.plot_crash_by_payload(runs, f"{OUT}/plots/crash_by_payload.png"); plt.show()
TEST_REFS = R.autopilot_reference(cfg.ENV, seed_base=1000, episodes=30, cache=f"{OUT}/autopilot_reference_test.json")
open(f"{OUT}/summary.md", "w").write(report.markdown(runs, REFS, TEST_REFS, CHECKS))
display(Markdown(report.markdown(runs, REFS, TEST_REFS, CHECKS)))

## 7. Watch them race
The best policies fly the **same three fresh test scenarios** side by side (same start, package, wind and gusts).
The trail turns red while a drone loses lift. MACURA's panel shows its own world model's verdict at every step:
**green = members agree** (MACURA would trust an imagined step here), **orange = they disagree** (it would stop
imagining); the strip underneath is the whole flight so far. Choose who races with `SHOW` (one algorithm or
several). Then a 30 s demo flight, MACURA vs MBPO (longer than the 10 s training flights; not a score).

In [ ]:
SHOW = ["macura", "mbpo", "sac"]      # one or several of the trained algorithms
BEST = {R.NAMES[a]: R.best_run(RUNS[a]) for a in SHOW if RUNS.get(a)}
PILOTS = {k: R.checkpoint(r) for k, r in BEST.items()}
TRUST = {k: trust_of(r) for k, r in BEST.items() if trust_of(r)}
if VIDEOS and PILOTS and hours_left() > 0.3:
    race = V.record_race(cfg.CFG, PILOTS, f"{OUT}/videos/race.mp4", seeds=(1000, 1001, 1002),
                         trust=TRUST, cols=min(len(PILOTS), 3), frame_step=3, fps=17)
    if race:
        display(Video(race, embed=True, width=960))

In [ ]:
if VIDEOS and "MACURA" in PILOTS and "MBPO" in PILOTS and hours_left() > 0.2:
    long = V.record_race(cfg.CFG, {k: PILOTS[k] for k in ("MACURA", "MBPO")}, f"{OUT}/videos/long_flight.mp4",
                         seeds=(1010, 1011), trust=TRUST, frame_step=3, fps=17, max_steps=1500)
    if long:
        display(Video(long, embed=True, width=960))

## 8. All seeds
For seed 2, add seed 1's output as an input (**Add Input → Your Work**, this notebook's seed-1 version) and this
section merges both seeds (only runs of the same length: the 50k pilot is kept apart). The same report is built on
your computer by `python compare.py out_seed1 out_seed2 --out results --videos`.

In [ ]:
ALL = R.load_runs(OUT, "/kaggle/input")
if len({r["seed"] for r in ALL}) > 1:
    made = report.build(ALL, cfg.CFG, f"{OUT}/all_seeds", check=True, videos=False, close=True)
    for name in ("learning_curves", "summary_table", "scores", "model_trust", "kappa", "model_check",
                 "untrusted_data"):
        if made.get(name):
            display(Image(made[name]))
    display(Markdown(open(made["summary"]).read()))
else:
    print("only this session's seed found: attach the other seed's output, or use compare.py on your computer")

## 9. Outputs
Everything is in `/kaggle/working/runs` and is saved as the notebook's output: `logs/` (one JSON per algorithm
and seed), `checkpoints/` (best policy `.zip`, its world model `_ensemble.pt`, `_meta.json`), `plots/`, `videos/`
(`test_<algo>.mp4`, `race.mp4`, `long_flight.mp4`), `summary.md`. Download each session into its own folder
(`out_seed1`, `out_seed2`), then on your computer:

    python compare.py out_seed1 out_seed2 --out results --videos
    mjpython simulate.py                          # all trained algorithms together
    mjpython simulate.py --algos macura           # one algorithm